#NER_Extraction

In [2]:
    import src.Project2.NER_Extraction.ner_extraction as NER_Extraction

    print("=== Step 1: Loading Macbeth ===")
    xml_content = NER_Extraction.load_and_clean(NER_Extraction.FILE_PATH)

    print("=== Step 2: Extracting dialogue scenes ===")
    scenes = NER_Extraction.extract_dialogue(xml_content)
    print(f"Found {len(scenes)} scenes")

    print("=== Step 3: Extracting official cast ===")
    characters = NER_Extraction.extract_cast(xml_content)
    print(f"Official characters: {characters}")

    print("=== Step 4: Running default spaCy NER ===")
    entities, summary = NER_Extraction.run_default_ner(scenes)
    print(f"Total entities found: {len(entities)}")

    print("=== Step 5: Analysing labels ===")
    mislabeled = NER_Extraction.analyse_labels(summary, characters)

    print("=== Step 6: Saving results ===")
    NER_Extraction.save_csv(entities, NER_Extraction.OUTPUT_CSV)

    print("\nDone. Review 01_default_ner_entities.csv for the full entity table.")
    print("Then run fine_tuning.py to correct the labels.")

=== Step 1: Loading Macbeth ===
=== Step 2: Extracting dialogue scenes ===
Found 28 scenes
=== Step 3: Extracting official cast ===
Official characters: ['DUNCAN', 'MALCOLM', 'DONALBAIN', 'MACBETH', 'BANQUO', 'MACDUFF', 'LENNOX', 'ROSS', 'MENTEITH', 'ANGUS', 'CAITHNESS', 'FLEANCE', 'SIWARD', 'YOUNG SIWARD', 'SEYTON', 'BOY', 'An English Doctor.', 'A Scottish Doctor.', 'A Soldier.', 'A Porter.', 'An Old Man.', 'LADY MACBETH.', 'LADY MACDUFF.', 'Gentlewoman attending on Lady Macbeth.', 'HECATE', 'Lords', 'The Ghost of Banquo and several other Apparitions.']
=== Step 4: Running default spaCy NER ===
Loading spaCy model: en_core_web_md ...
Total entities found: 750
=== Step 5: Analysing labels ===

=== LABEL SUMMARY (default model) ===

[CARDINAL] (17 unique):
   One
   Two
   accus'd
   eight
   enough.—Come
   half
   nine
   one
   supp'd
   ten
   ten thousand
   thirty-one
   thousands
   three
   twelve
   ... and 2 more

[DATE] (21 unique):
   Days
   May
   SECOND
   Thou'lt
   Tomo

#Fine-Tuning

In [ ]:
    import spacy
    import src.Project2.Fine_Tuning.fine_tuning as Fine_Tuning
    from pathlib import Path

    print("=== Loading and parsing Macbeth ===")
    xml_content = Fine_Tuning.load_and_clean(Fine_Tuning.FILE_PATH)
    characters  = Fine_Tuning.extract_cast(xml_content)
    scenes      = Fine_Tuning.extract_scenes(xml_content)
    print(f"Characters: {len(characters)}  |  Scenes: {len(scenes)}")
    print(f"Character list: {characters}")

    # Need a plain nlp for make_doc (before training)
    nlp_base = spacy.load("en_core_web_md")

    print("\n=== Building training data ===")
    examples = Fine_Tuning.build_training_data(scenes, characters, nlp_base)

    print("\n=== Fine-tuning model ===")
    nlp_finetuned = Fine_Tuning.fine_tune(examples, n_iter=40)

    print("\n=== Saving fine-tuned model ===")
    Path(Fine_Tuning.MODEL_OUT).mkdir(parents=True, exist_ok=True)
    nlp_finetuned.to_disk(Fine_Tuning.MODEL_OUT)
    print(f"Model saved → {Fine_Tuning.MODEL_OUT}")

    print("\n=== Extracting entities with fine-tuned model ===")
    records, summary = Fine_Tuning.extract_finetuned_entities(nlp_finetuned, scenes)

    print("\n=== Entity Summary (fine-tuned) ===")
    for label, ents in sorted(summary.items()):
        print(f"\n[{label}] — {len(ents)} unique entities:")
        for e in sorted(ents)[:20]:
            print(f"   {e}")

    Fine_Tuning.save_csv(records, Fine_Tuning.OUTPUT_CSV)
    print("\nDone. Fine-tuned entities saved. Run 03_coreference.py next.")

=== Loading and parsing Macbeth ===
Characters: 25  |  Scenes: 56
Character list: ['DUNCAN', 'MALCOLM', 'DONALBAIN', 'MACBETH', 'BANQUO', 'MACDUFF', 'LENNOX', 'ROSS', 'MENTEITH', 'ANGUS', 'CAITHNESS', 'FLEANCE', 'SIWARD', 'YOUNG SIWARD', 'SEYTON', 'BOY', 'An English Doctor.', 'A Scottish Doctor.', 'An Old Man.', 'LADY MACBETH.', 'LADY MACDUFF.', 'Gentlewoman attending on Lady Macbeth.', 'HECATE', 'Lords', 'The Ghost of Banquo and several other Apparitions.']

=== Building training data ===


C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "That now Sweno, the Norways' king, craves composit..." with entities "[(20, 26, 'GPE'), (117, 135, 'GPE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "How far is't call'd to Forres?—What are these, So ..." with entities "[(23, 29, 'GPE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: User

Built 330 training examples (0 skipped due to alignment errors)

=== Fine-tuning model ===
Loading base model: en_core_web_md ...
